### HYDRUS Automation Process
#### This notebook will take in input of vG parameters and load them into HYDRUS, and then run a drainage model for the soil profile. This inital automation will be for a singular set of inputs and can be expanded for a range later.

In [329]:
from pathlib import Path
import shutil
import subprocess
import phydrus as ps
from pywinauto import Desktop
import time
from pywinauto.keyboard import send_keys
import pyperclip



Notebook condensed into blocks and functions that are not commented out. Pieces exsist in the comments and archive if needed in the future.

##### Current workflow:
    1. Define van Genuchten parameters in Python.
    2. Write parameters into SELECTOR.IN.
    3. Create LEVEL_01.DIR pointer file.
    4. Run HYDRUS-1D solver (H1D_UNSC.EXE).
    5. Verify output files update.

    Next objective:

    6. Read HYDRUS output files into Python.
    7. Extract θ(z,t) profiles.
    8. Convert θ to electrical conductivity.
    9. Run TEM forward model.

### 1. Define paths

In [330]:
install_dir = Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx")
project_dir = Path(r"C:\TDEM HYDRUS\Drainage - SHARP")
runner_dir = Path(r"C:\TDEM HYDRUS\hydrus_runner")

selector_file = project_dir / "SELECTOR.IN"
solver_src = install_dir / "H1D_UNSC.EXE"
solver_dst = runner_dir / "H1D_UNSC.EXE"

project_file = r"C:\TDEM HYDRUS\Drainage - SHARP.h1d"

### 2. Define vG parameters for HYDRUS input file

In [331]:
vg_params = {
    "theta_r": 0.045,
    "theta_s": 0.430,
    "alpha": 0.145,
    "n": 2.680,
    "Ks": 812.8,
    "l": 0.5
} 

### 3. Write vG parameters into the input file

In [332]:
""" ## Assign new parameters to the SELECTOR.IN file

new_line = (
    f"{vg_params['theta_r']:>8.3f}"
    f"{vg_params['theta_s']:>8.3f}"
    f"{vg_params['alpha']:>8.3f}"
    f"{vg_params['n']:>8.3f}"
    f"{vg_params['Ks']:>12.4f}"
    f"{vg_params['l']:>8.3f}"
)

text = selector_file.read_text()
lines = text.splitlines()

for i, line in enumerate(lines):
    if "thr" in line and "ths" in line and "Alfa" in line:
        lines[i + 1] = new_line
        break

selector_file.write_text("\n".join(lines))

print("Updated HYDRUS parameters:")
print(new_line)""" 

' ## Assign new parameters to the SELECTOR.IN file\n\nnew_line = (\n    f"{vg_params[\'theta_r\']:>8.3f}"\n    f"{vg_params[\'theta_s\']:>8.3f}"\n    f"{vg_params[\'alpha\']:>8.3f}"\n    f"{vg_params[\'n\']:>8.3f}"\n    f"{vg_params[\'Ks\']:>12.4f}"\n    f"{vg_params[\'l\']:>8.3f}"\n)\n\ntext = selector_file.read_text()\nlines = text.splitlines()\n\nfor i, line in enumerate(lines):\n    if "thr" in line and "ths" in line and "Alfa" in line:\n        lines[i + 1] = new_line\n        break\n\nselector_file.write_text("\n".join(lines))\n\nprint("Updated HYDRUS parameters:")\nprint(new_line)'

## 4. Automate HYDRUS run

4.1 One full function to open and run HYDRUS for use in the circular modeling process

In [333]:
### One function to automate the HYDRUS run process for future use
### Opens HYDRUS, loads the project file, and executes the run

def run_hydrus(hydrus_gui, project_file,vg_params, wait_after_run=10):

    hydrus_gui = Path(hydrus_gui)
    project_file = Path(project_file)

    ### Update HYDRUS files 
    ## Assign new parameters to the SELECTOR.IN file

    new_line = (
        f"{vg_params['theta_r']:>8.3f}"
        f"{vg_params['theta_s']:>8.3f}"
        f"{vg_params['alpha']:>8.3f}"
        f"{vg_params['n']:>8.3f}"
        f"{vg_params['Ks']:>12.4f}"
        f"{vg_params['l']:>8.3f}"
    )

    text = selector_file.read_text()
    lines = text.splitlines()

    for i, line in enumerate(lines):
        if "thr" in line and "ths" in line and "Alfa" in line:
            lines[i + 1] = new_line
            break

    selector_file.write_text("\n".join(lines))

    print("Updated HYDRUS parameters:")
    print(new_line)

    ### Open HYDRUS GUI and detect window
    process = subprocess.Popen(
        [str(hydrus_gui)],
        cwd=hydrus_gui.parent
    )

    time.sleep(wait_after_run) # give HYDRUS time to open

    ### Find the HYDRUS window
    def get_hydrus_window():
        windows = Desktop(backend="uia").windows()

        for w in windows:
            title = w.window_text()

        # Select only the actual HYDRUS program window
            if title == "HYDRUS-1D" or title.startswith("HYDRUS-1D -"):
                return w

        raise RuntimeError("HYDRUS window not found.")

    
    ### Open HYDRUS and file window
    hydrus_window = get_hydrus_window()
    hydrus_window.set_focus()
    time.sleep(1)

    print("Focused window:", hydrus_window.window_text())

    send_keys("%f")
    send_keys("o")

    ### Paste file name and open project
    hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D.*")
    open_dialog = hydrus_app.child_window(title="Open", control_type="Window")

    open_dialog.set_focus()
    time.sleep(0.5)

    pyperclip.copy(project_file)

    send_keys("%n")      # focus File name box
    time.sleep(0.5)

    send_keys("^v")      # paste path
    time.sleep(0.5)

    send_keys("{ENTER}") # open
    time.sleep(2)

    ### Open calculation menu and execute run
    hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D - Drainage - SHARP.*")
    hydrus_app.set_focus()
    time.sleep(0.5)

    send_keys("%c")
    time.sleep(0.5)

    items = hydrus_app.descendants(title="Execute HYDRUS", control_type="MenuItem")

    print(len(items))
    for item in items:
        print(item)

    items[0].click_input()

    ### Hit okay to start the run 
    time.sleep(0.5)
    send_keys("{ENTER}")

    ### Wait and enter to finish
    time.sleep(wait_after_run)
    send_keys("{ENTER}")

    return process

In [334]:
### Call the function to run HYDRUS

run_hydrus(
    hydrus_gui=Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx\Hydrus1D.exe"),
    project_file=Path(r"C:\TDEM HYDRUS\Drainage - SHARP.h1d"), vg_params=vg_params,
    wait_after_run=10
)

Updated HYDRUS parameters:
   0.045   0.430   0.145   2.680    812.8000   0.500
Focused window: HYDRUS-1D - Drainage - SHARP
2
uia_controls.MenuItemWrapper - 'Execute HYDRUS', MenuItem
uia_controls.MenuItemWrapper - 'Execute HYDRUS', MenuItem


<Popen: returncode: None args: ['C:\\Program Files (x86)\\PC-Progress\\Hydru...>

Open HYDRUS and Detect window  note: necessary for function

In [335]:
""" hydrus_gui = Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx\Hydrus1D.exe")

process = subprocess.Popen(
    [str(hydrus_gui)],
    cwd=hydrus_gui.parent
)

time.sleep(5)

windows = Desktop(backend="uia").windows()

for w in windows:
    title = w.window_text()
    if "HYDRUS" in title.upper():
        print("Found HYDRUS window:")
        print(title) """

' hydrus_gui = Path(r"C:\\Program Files (x86)\\PC-Progress\\Hydrus-1D 4.xx\\Hydrus1D.exe")\n\nprocess = subprocess.Popen(\n    [str(hydrus_gui)],\n    cwd=hydrus_gui.parent\n)\n\ntime.sleep(5)\n\nwindows = Desktop(backend="uia").windows()\n\nfor w in windows:\n    title = w.window_text()\n    if "HYDRUS" in title.upper():\n        print("Found HYDRUS window:")\n        print(title) '

Find HYDRUS-1D

In [336]:
""" def get_hydrus_window():
    windows = Desktop(backend="uia").windows()

    for w in windows:
        title = w.window_text()

        # Select only the actual HYDRUS program window
        if title == "HYDRUS-1D" or title.startswith("HYDRUS-1D -"):
            return w

    raise RuntimeError("HYDRUS window not found.") """

' def get_hydrus_window():\n    windows = Desktop(backend="uia").windows()\n\n    for w in windows:\n        title = w.window_text()\n\n        # Select only the actual HYDRUS program window\n        if title == "HYDRUS-1D" or title.startswith("HYDRUS-1D -"):\n            return w\n\n    raise RuntimeError("HYDRUS window not found.") '

Open HYDRUS and file menu

In [337]:
"""hydrus_window = get_hydrus_window()
hydrus_window.set_focus()
time.sleep(1)

print("Focused window:", hydrus_window.window_text())

send_keys("%f")
send_keys("o")""" 

'hydrus_window = get_hydrus_window()\nhydrus_window.set_focus()\ntime.sleep(1)\n\nprint("Focused window:", hydrus_window.window_text())\n\nsend_keys("%f")\nsend_keys("o")'

Paste file name and open

In [338]:
"""hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D.*")
open_dialog = hydrus_app.child_window(title="Open", control_type="Window")

open_dialog.set_focus()
time.sleep(0.5)

project_file = r"C:\TDEM HYDRUS\Drainage - SHARP.h1d"
pyperclip.copy(project_file)

send_keys("%n")      # focus File name box
time.sleep(0.5)

send_keys("^v")      # paste path
time.sleep(0.5)

send_keys("{ENTER}") # open"""

'hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D.*")\nopen_dialog = hydrus_app.child_window(title="Open", control_type="Window")\n\nopen_dialog.set_focus()\ntime.sleep(0.5)\n\nproject_file = r"C:\\TDEM HYDRUS\\Drainage - SHARP.h1d"\npyperclip.copy(project_file)\n\nsend_keys("%n")      # focus File name box\ntime.sleep(0.5)\n\nsend_keys("^v")      # paste path\ntime.sleep(0.5)\n\nsend_keys("{ENTER}") # open'

Open Calculation menu and hit Execute HYDRUS

In [339]:
"""hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D - Drainage - SHARP.*")
hydrus_app.set_focus()
time.sleep(0.5)

send_keys("%c")
time.sleep(0.5)

items = hydrus_app.descendants(title="Execute HYDRUS", control_type="MenuItem")

print(len(items))
for item in items:
    print(item)

items[0].click_input()"""

'hydrus_app = Desktop(backend="uia").window(title_re=".*HYDRUS-1D - Drainage - SHARP.*")\nhydrus_app.set_focus()\ntime.sleep(0.5)\n\nsend_keys("%c")\ntime.sleep(0.5)\n\nitems = hydrus_app.descendants(title="Execute HYDRUS", control_type="MenuItem")\n\nprint(len(items))\nfor item in items:\n    print(item)\n\nitems[0].click_input()'

Hit ok to start the run

In [340]:
"""time.sleep(0.5)
send_keys("{ENTER}")"""

'time.sleep(0.5)\nsend_keys("{ENTER}")'

Hit enter to finish

In [341]:
"""# give HYDRUS time to finish
time.sleep(5)

send_keys("{ENTER}")"""

'# give HYDRUS time to finish\ntime.sleep(5)\n\nsend_keys("{ENTER}")'

### 5. Read HYDRUS Outputs 


In [342]:
"""# ============================================================
# DIAGNOSTIC: List HYDRUS output files and sizes
# ============================================================

for file in project_dir.glob("*.OUT"):
    print(file.name, file.stat().st_size)"""

'# ============================================================\n# DIAGNOSTIC: List HYDRUS output files and sizes\n# ============================================================\n\nfor file in project_dir.glob("*.OUT"):\n    print(file.name, file.stat().st_size)'

In [343]:
"""# ============================================================
# DIAGNOSTIC: Inspect PROFILE.OUT
# ============================================================

profile_out_file = project_dir / "PROFILE.OUT"

with open(profile_out_file, "r", errors="ignore") as f:
    for i in range(120):
        print(f.readline().rstrip())"""

'# ============================================================\n# DIAGNOSTIC: Inspect PROFILE.OUT\n# ============================================================\n\nprofile_out_file = project_dir / "PROFILE.OUT"\n\nwith open(profile_out_file, "r", errors="ignore") as f:\n    for i in range(120):\n        print(f.readline().rstrip())'

In [344]:
"""nod_inf_file = project_dir / "NOD_INF.OUT"

nod_inf = ps.read_nod_inf(nod_inf_file)

print(type(nod_inf))
print(nod_inf.keys())"""

'nod_inf_file = project_dir / "NOD_INF.OUT"\n\nnod_inf = ps.read_nod_inf(nod_inf_file)\n\nprint(type(nod_inf))\nprint(nod_inf.keys())'

ARCHIVE FOR LATER 

In [345]:
""" # ============================================================
# ARCHIVE / FUTURE USE: Create a fresh HYDRUS run folder
# ============================================================

from pathlib import Path
import shutil

template_dir = Path(r"C:\TDEM HYDRUS\Drainage - SHARP")
runs_dir = Path(r"C:\TDEM HYDRUS\HYDRUS_Runs")
run_dir = runs_dir / "run_001"

runs_dir.mkdir(exist_ok=True)

if run_dir.exists():
    shutil.rmtree(run_dir)

shutil.copytree(template_dir, run_dir)

print(f"Created new HYDRUS run folder: {run_dir}") """

' # ============================================================\n# ARCHIVE / FUTURE USE: Create a fresh HYDRUS run folder\n# ============================================================\n\nfrom pathlib import Path\nimport shutil\n\ntemplate_dir = Path(r"C:\\TDEM HYDRUS\\Drainage - SHARP")\nruns_dir = Path(r"C:\\TDEM HYDRUS\\HYDRUS_Runs")\nrun_dir = runs_dir / "run_001"\n\nruns_dir.mkdir(exist_ok=True)\n\nif run_dir.exists():\n    shutil.rmtree(run_dir)\n\nshutil.copytree(template_dir, run_dir)\n\nprint(f"Created new HYDRUS run folder: {run_dir}") '

In [346]:
""" # ============================================================
# ARCHIVE / DEBUGGING: Check whether HYDRUS output files updated
# ============================================================

from pathlib import Path
import os
import time

project_dir = Path(r"C:\TDEM HYDRUS\Drainage - SHARP")

output_files = [
    "NOD_INF.OUT",
    "T_LEVEL.OUT",
    "BALANCE.OUT",
    "Profile.out"
]

for filename in output_files:
    file_path = project_dir / filename

    if file_path.exists():
        modified_time = time.ctime(os.path.getmtime(file_path))
        print(f"{filename}: {modified_time}")
    else:
        print(f"{filename}: not found") """

' # ============================================================\n# ARCHIVE / DEBUGGING: Check whether HYDRUS output files updated\n# ============================================================\n\nfrom pathlib import Path\nimport os\nimport time\n\nproject_dir = Path(r"C:\\TDEM HYDRUS\\Drainage - SHARP")\n\noutput_files = [\n    "NOD_INF.OUT",\n    "T_LEVEL.OUT",\n    "BALANCE.OUT",\n    "Profile.out"\n]\n\nfor filename in output_files:\n    file_path = project_dir / filename\n\n    if file_path.exists():\n        modified_time = time.ctime(os.path.getmtime(file_path))\n        print(f"{filename}: {modified_time}")\n    else:\n        print(f"{filename}: not found") '

In [347]:
""" # ============================================================
# ARCHIVE / DEBUGGING: List HYDRUS executables
# ============================================================

from pathlib import Path

hydrus_install_dir = Path(r"C:\Program Files (x86)\PC-Progress\Hydrus-1D 4.xx")

for file in hydrus_install_dir.glob("*.EXE"):
    print(file.name) """

' # ============================================================\n# ARCHIVE / DEBUGGING: List HYDRUS executables\n# ============================================================\n\nfrom pathlib import Path\n\nhydrus_install_dir = Path(r"C:\\Program Files (x86)\\PC-Progress\\Hydrus-1D 4.xx")\n\nfor file in hydrus_install_dir.glob("*.EXE"):\n    print(file.name) '

In [348]:
""" 
# ============================================================
# ARCHIVE / DEBUGGING: Create Level_01.DIR pointer file
# ============================================================
### Use Python to create a .DIR file for HYDRUS that points to the project directory
project_dir = Path(r"C:\TDEM HYDRUS\Drainage - SHARP")
dir_file = project_dir / "LEVEL_01.DIR"

dir_file.write_text(r"C:\TDEM HYDRUS\Drainage - SHARP\\")

print("Created:")
print(dir_file)
print("Contents:")
print(dir_file.read_text()) """

' \n# ============================================================\n# ARCHIVE / DEBUGGING: Create Level_01.DIR pointer file\n# ============================================================\n### Use Python to create a .DIR file for HYDRUS that points to the project directory\nproject_dir = Path(r"C:\\TDEM HYDRUS\\Drainage - SHARP")\ndir_file = project_dir / "LEVEL_01.DIR"\n\ndir_file.write_text(r"C:\\TDEM HYDRUS\\Drainage - SHARP\\")\n\nprint("Created:")\nprint(dir_file)\nprint("Contents:")\nprint(dir_file.read_text()) '